In [21]:
import os
import glob
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [22]:
# Cost isn't a concern since we're using free tiers, but we do need multiple
# provider keys since we're spreading calls across Gemini/Groq/OpenRouter

db_name = "vector_db"

load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set")

Gemini API Key exists and begins AQ.Ab8RN
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-


In [23]:
from huggingface_hub import login

hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub")
else:
    print("HF_TOKEN not set — downloads may be slower/rate-limited")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face Hub


In [24]:
load_dotenv(override=True)

KNOWLEDGE_BASE_PATH = "../knowledge-base"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

In [25]:
# Find every category subfolder inside knowledge-base/ 
# (players, formats, tournaments, icc)
folders = glob.glob(f"{KNOWLEDGE_BASE_PATH}/*")

documents = []
for folder in folders:
    # folder looks like "../knowledge-base/players" — grab just "players"
    # so we can tag every document from this folder with its category
    category = folder.split("/")[-1]

    # DirectoryLoader reads every .md file in this one folder.
    # TextLoader is the actual file reader it uses per file — just plain text.
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",              # only pick up markdown files
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}  # avoid encoding errors on load
    )
    folder_docs = loader.load()

    # Each loaded document has a .metadata dict (currently just holds
    # the file path). We add "category" to it manually, so after all
    # documents get mixed together and later split into chunks, we can
    # still tell which category any given chunk came from.
    for doc in folder_docs:
        doc.metadata["category"] = category

    # Add this folder's documents to our running master list
    documents.extend(folder_docs)

print(f"Loaded {len(documents)} documents")

Loaded 17 documents


In [26]:
# How many characters in all the documents?

knowledge_base_path = "../knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 17 files in the knowledge base
Total characters in knowledge base: 30,001


In [27]:
# Gemini's own SDK can count tokens exactly for whichever Gemini model
# we specify — free, no extra cost, and accurate for the model we're
# actually using (unlike tiktoken, which only knows OpenAI's tokenization).

from google import genai

gemini_client = genai.Client(api_key=gemini_api_key)

response = gemini_client.models.count_tokens(
    model="gemini-3.6-flash",
    contents=entire_knowledge_base
)

print(f"Total tokens (Gemini gemini-3.6-flash tokenizer): {response.total_tokens:,}")

Total tokens (Gemini gemini-3.6-flash tokenizer): 7,251


In [28]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("../knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 17 documents


In [29]:
documents[1]

Document(metadata={'source': '../knowledge-base/formats/t20i-cricket.md', 'doc_type': 'formats'}, page_content="# Twenty20 International (T20I) Cricket\n\nTwenty20 International cricket is the shortest and most fast-paced format of international cricket, designed to deliver a complete match in around three hours, and has become the most commercially popular format of the modern game.\n\n## T20I Format Rules\n\n- In T20I cricket, each team bats for a maximum of **20 overs** (120 balls) per innings.\n- T20I format prioritizes aggressive batting, with fielding restrictions during a mandatory powerplay of the first 6 overs, limiting fielders outside the inner circle.\n- A Super Over is used to decide matches that finish level, including in knockout stages of major tournaments.\n- Matches are typically completed within roughly 3 hours, making the format highly broadcast-friendly.\n\n## T20I Cricket Key Characteristics\n\n- T20I cricket was first played as an international match in 2005 (Aus

In [30]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 38 chunks
First chunk:

page_content='# Test Cricket

Test cricket is the longest format of the sport and is widely regarded as the pinnacle discipline of the game, testing a team's skill, patience, and endurance over an extended contest.

## Test Cricket Format Rules

- Test matches are played over a maximum of **5 days**, with up to 90 overs bowled per day (six balls per over).
- Each team bats twice (two innings), unless the match is decided earlier via a follow-on or the match ends in a draw or tie.
- There is no limit on the number of overs a team can bat for in an innings — a team bats until it is bowled out (10 wickets fall) or chooses to declare its innings closed.
- A match can end in a **win, loss, draw** (time runs out with the match unresolved), or extremely rarely, a **tie** (only two ties in Test history).
- The follow-on rule allows a team leading by 200+ runs after the first innings to force the opposition to bat again immediately.

## Test Cricket Key Char

In [31]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Split on markdown headers first — this respects our documents' actual
# structure (# Title, ## Section) rather than just cutting by character count

headers_to_split_on = [
    ("#", "header_1"),
    ("##", "header_2"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

markdown_chunks = []
for doc in documents:
    doc_chunks = markdown_splitter.split_text(doc.page_content)
    # MarkdownHeaderTextSplitter strips the original metadata (source, doc_type),
    # so we need to re-attach it to every chunk it produces
    for chunk in doc_chunks:
        chunk.metadata.update(doc.metadata)
    markdown_chunks.extend(doc_chunks)

print(f"RecursiveCharacterTextSplitter: {len(chunks)} chunks")
print(f"MarkdownHeaderTextSplitter: {len(markdown_chunks)} chunks")

RecursiveCharacterTextSplitter: 38 chunks
MarkdownHeaderTextSplitter: 81 chunks


In [32]:
recursive_lengths = [len(c.page_content) for c in chunks]
markdown_lengths = [len(c.page_content) for c in markdown_chunks]

print("RecursiveCharacterTextSplitter chunk lengths:")
print(f"  min={min(recursive_lengths)}, max={max(recursive_lengths)}, avg={sum(recursive_lengths)/len(recursive_lengths):.0f}")

print("\nMarkdownHeaderTextSplitter chunk lengths:")
print(f"  min={min(markdown_lengths)}, max={max(markdown_lengths)}, avg={sum(markdown_lengths)/len(markdown_lengths):.0f}")

RecursiveCharacterTextSplitter chunk lengths:
  min=222, max=990, avg=806

MarkdownHeaderTextSplitter chunk lengths:
  min=49, max=708, avg=342


In [13]:
for c in markdown_chunks[:3]:
    print(c.metadata)
    print(c.page_content[:200])
    print("---")

{'header_1': 'Test Cricket', 'source': '../knowledge-base/formats/test-cricket.md', 'doc_type': 'formats'}
Test cricket is the longest format of the sport and is widely regarded as the pinnacle discipline of the game, testing a team's skill, patience, and endurance over an extended contest.
---
{'header_1': 'Test Cricket', 'header_2': 'Format Rules', 'source': '../knowledge-base/formats/test-cricket.md', 'doc_type': 'formats'}
- Matches are played over a maximum of **5 days**, with up to 90 overs bowled per day (six balls per over).
- Each team bats twice (two innings), unless the match is decided earlier via a follow-on or
---
{'header_1': 'Test Cricket', 'header_2': 'Key Characteristics', 'source': '../knowledge-base/formats/test-cricket.md', 'doc_type': 'formats'}
- Players wear traditional white clothing and use a red (or, in day-night Tests, pink) ball.
- Considered the ultimate test of technique, temperament, and fitness due to its length.
- The **ICC World 
---


In [33]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings

candidate_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "BAAI/bge-small-en-v1.5",
]

embedding_results = {}

# Test the three local HuggingFace models
for model_name in candidate_models:
    print(f"Testing {model_name}...")
    start = time.time()

    test_embeddings = HuggingFaceEmbeddings(model_name=model_name)
    test_store = Chroma.from_documents(
        documents=markdown_chunks,
        embedding=test_embeddings,
        collection_name=f"test_{model_name.replace('/', '_')}"
    )

    elapsed = time.time() - start
    sample_vec = test_store._collection.get(limit=1, include=["embeddings"])["embeddings"][0]

    embedding_results[model_name] = {
        "store": test_store,
        "time": elapsed,
        "dimensions": len(sample_vec)
    }

    print(f"  Time: {elapsed:.1f}s | Dimensions: {len(sample_vec)}")
    print()

# Test Gemini's hosted embedding API — different class, same result shape,
# so it slots into the same embedding_results dict as the others
print("Testing gemini-embedding-001...")
start = time.time()

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=gemini_api_key  # this API reads GOOGLE_API_KEY by default, so we pass ours explicitly
)
gemini_store = Chroma.from_documents(
    documents=markdown_chunks,
    embedding=gemini_embeddings,
    collection_name="test_gemini_embedding"
)

elapsed = time.time() - start
sample_vec = gemini_store._collection.get(limit=1, include=["embeddings"])["embeddings"][0]

embedding_results["gemini-embedding-001"] = {
    "store": gemini_store,
    "time": elapsed,
    "dimensions": len(sample_vec)
}

print(f"  Time: {elapsed:.1f}s | Dimensions: {len(sample_vec)}")

Testing all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8025.51it/s]


  Time: 10.4s | Dimensions: 384

Testing all-mpnet-base-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8346.25it/s]


  Time: 10.4s | Dimensions: 768

Testing BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8132.46it/s]


  Time: 8.9s | Dimensions: 384

Testing gemini-embedding-001...
  Time: 5.2s | Dimensions: 3072


In [34]:
test_queries = [
    "Who has the record for most ODI centuries?",
    "How many overs are bowled per day in a Test match?",
    "Which player was named Player of the Match in the 2026 T20 World Cup final?",
    # Tests whether the model confuses two formats with similar-sounding rules
    "Which cricket format uses fielding restrictions during a powerplay of the first 6 overs?",
    
    # Requires distinguishing a precise number among several similar numbers in the same document
    "What is Don Bradman's exact career Test batting average?",
    
    # Multi-entity: tests whether retrieval finds the specific franchise connection, not just any IPL mention
    "Which IPL franchise has Jasprit Bumrah played for since 2013?",
    
    # Tests negative/contrastive reasoning — the answer requires knowing what does NOT apply
    "Which cricket format does not have a follow-on rule?",
    
    # Precise historic detail buried in a longer paragraph, not the first sentence
    "In which year and city was the first-ever Test match played?",
]
for model_name, result in embedding_results.items():
    print(f"=== {model_name} ===")
    for query in test_queries:
        top_result = result["store"].similarity_search(query, k=1)[0]
        print(f"Q: {query}")
        # print(f"   -> {top_result.page_content[:150]}")
        print(f"   -> {top_result.page_content}")
    print()

=== all-MiniLM-L6-v2 ===
Q: Who has the record for most ODI centuries?
   -> - The world record for the highest individual ODI score is 264, set by Rohit Sharma.
- Virat Kohli holds the record for most ODI centuries (54), a milestone he reached during the 2023 Cricket World Cup, surpassing Sachin Tendulkar's previous record of 49.
Q: How many overs are bowled per day in a Test match?
   -> - Test matches are played over a maximum of **5 days**, with up to 90 overs bowled per day (six balls per over).
- Each team bats twice (two innings), unless the match is decided earlier via a follow-on or the match ends in a draw or tie.
- There is no limit on the number of overs a team can bat for in an innings — a team bats until it is bowled out (10 wickets fall) or chooses to declare its innings closed.
- A match can end in a **win, loss, draw** (time runs out with the match unresolved), or extremely rarely, a **tie** (only two ties in Test history).
- The follow-on rule allows a team leading by

In [17]:
# Prework — extract vectors, documents, and doc_types for each model's store

viz_data = {}

for model_name, result in embedding_results.items():
    collection = result["store"]._collection
    data = collection.get(include=['embeddings', 'documents', 'metadatas'])
    
    vectors = np.array(data['embeddings'])
    docs = data['documents']
    metadatas = data['metadatas']
    doc_types = [m['doc_type'] for m in metadatas]
    
    # our four categories, matching knowledge-base folder names
    categories = ['players', 'formats', 'tournaments', 'icc']
    colors = [['blue', 'green', 'red', 'orange'][categories.index(t)] for t in doc_types]
    
    viz_data[model_name] = {
        'vectors': vectors,
        'documents': docs,
        'doc_types': doc_types,
        'colors': colors
    }

print("Prework done for:", list(viz_data.keys()))

Prework done for: ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'BAAI/bge-small-en-v1.5', 'gemini-embedding-001']


In [18]:
for model_name, data in viz_data.items():
    tsne = TSNE(n_components=2, random_state=42)
    reduced_vectors = tsne.fit_transform(data['vectors'])

    fig = go.Figure(data=[go.Scatter(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        mode='markers',
        marker=dict(size=5, color=data['colors'], opacity=0.8),
        text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(data['doc_types'], data['documents'])],
        hoverinfo='text'
    )])

    fig.update_layout(
        title=f'2D Vector Store Visualization — {model_name}',
        xaxis_title='x',
        yaxis_title='y',
        width=800,
        height=600,
        margin=dict(r=20, b=10, l=10, t=40)
    )

    fig.show()

In [19]:
for model_name, data in viz_data.items():
    tsne = TSNE(n_components=3, random_state=42)
    reduced_vectors = tsne.fit_transform(data['vectors'])

    fig = go.Figure(data=[go.Scatter3d(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        z=reduced_vectors[:, 2],
        mode='markers',
        marker=dict(size=5, color=data['colors'], opacity=0.8),
        text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(data['doc_types'], data['documents'])],
        hoverinfo='text'
    )])

    fig.update_layout(
        title=f'3D Vector Store Visualization — {model_name}',
        scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
        width=900,
        height=700,
        margin=dict(r=10, b=10, l=10, t=40)
    )

    fig.show()

In [35]:
# Let's investigate the vectors

collection = embedding_results["all-MiniLM-L6-v2"]["store"]._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 162 vectors with 384 dimensions in the vector store


In [ ]:
import google
print(google.__path__)

import sys
print(sys.executable)

In [ ]:
# metadata should show something like:
# {'source': '../knowledge-base/players/ben-stokes.md', 'category': 'players'}
print(documents[0].metadata)

# page_content is the raw text of the whole file — uncut, unsplit.
# We print only the first 200 characters just to sanity-check it loaded right.
print(documents[0].page_content[:200])

In [ ]:
# RecursiveCharacterTextSplitter breaks long documents into smaller pieces.
# "Recursive" means it tries to split on natural boundaries first (paragraphs,
# then sentences, then words) rather than cutting mid-sentence wherever the
# character count happens to land.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # target size per chunk, in characters
    chunk_overlap=200     # each chunk repeats the last 200 characters of the
                           # previous one, so context isn't lost at the cut point
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")

In [ ]:
print(chunks[0].metadata)
print(chunks[0].page_content)
print(f"\nChunk length: {len(chunks[0].page_content)} characters")